# 03 · Cross-check sources and prepare manual checks

1. **Cross-source comparison** where two sources cover the same election: 2002 (france-politique vs
   Wikipedia) and 2022 (NSPPolls vs Wikipedia). Polls are matched on pollster and fieldwork dates, scenarios
   on their candidate lists, then shares are compared candidate by candidate.
2. **Flagged overlaps** from notebook 02 (`overlap_client_unknown`): resolved automatically for 2002 and
   2022 from the source that records the client; 2017 goes to a manual worklist.
3. **Random spot-check sample**: 30 Wikipedia polls per election, stratified by pollster, drawn with the
   seed in `config/validation.yaml`, to check against the original documents.

Worklists go to `validation/` and are never overwritten once they exist: fill them in there.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import yaml
from polls import crosscheck, sampling, worklists

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
INTERIM = ROOT / "data" / "interim"
VALIDATION = ROOT / "validation"

wp = pd.read_csv(INTERIM / "polls_wikipedia.csv")
fp = pd.read_csv(INTERIM / "polls_france_politique.csv")
nsp = pd.read_csv(INTERIM / "polls_nsppolls.csv")
thinned = pd.read_csv(INTERIM / "thinning.csv")
thinned["overlap_client_unknown"] = thinned["overlap_client_unknown"].astype(bool)
thinned["kept"] = thinned["kept"].astype(bool)

## 1 · Cross-source comparison

Verdicts per candidate: `equal`; `differ` (both give a number, different); `compatible (threshold)` (`<0,5 %`
on one side, 0 or 0.5 on the other); `status differs`; `candidate only in a/b` (a number on one side, absent
on the other); `candidate only in a/b, without a share` (e.g. marked `autres` on france-politique and not
listed on Wikipedia: a coverage difference, not an error).

Source *a* is france-politique (2002) or NSPPolls (2022); source *b* is Wikipedia.

In [2]:
TO_CHECK = {"differ", "status differs", "candidate only in a", "candidate only in b",
            "scenario only in a", "scenario only in b"}
pairs = {2002: (fp, wp[wp["election"] == 2002]), 2022: (nsp, wp[wp["election"] == 2022])}
matches, shares = {}, {}
for year, (a, b) in pairs.items():
    matches[year] = crosscheck.match_polls(a, b)
    shares[year] = crosscheck.compare_shares(a, b, matches[year])
    matches[year].to_csv(INTERIM / f"crosscheck_{year}_polls.csv", index=False)
    shares[year].to_csv(INTERIM / f"crosscheck_{year}_shares.csv", index=False)

pd.concat({y: m["status"].value_counts() for y, m in matches.items()}, axis=1).fillna(0).astype(int)

,2002,2022
status,,
matched,97,239
only in b,1,16
only in a,1,104
"matched, dates differ",0,8


In [3]:
pd.concat({y: s["verdict"].value_counts() for y, s in shares.items()}, axis=1).fillna(0).astype(int)

,2002,2022
verdict,,
equal,1377,5398
"candidate only in a, without a share",58,0
differ,17,97
candidate only in a,13,17
status differs,2,5
compatible (threshold),0,270
candidate only in b,0,14
scenario only in a,0,2


In [4]:
for year, s in shares.items():
    d = s[s["verdict"] == "differ"]
    gap = (d["share_a"] - d["share_b"]).abs()
    print(f"{year}: {len(d)} differing shares in {d['poll_id_a'].nunique()} polls; "
          f"gap median {gap.median():.1f}, max {gap.max():.1f} points")

2002: 17 differing shares in 9 polls; gap median 0.5, max 2.0 points
2022: 97 differing shares in 37 polls; gap median 1.0, max 4.0 points


Every scenario with at least one disagreement goes to `validation/crosscheck_disagreements.csv`, one row per
scenario, to settle against the original document (Commission des sondages notice or pollster report).
Each row says how to find the scenario in the document: its position in the poll (`s2 of 5`), the
source's own label if any (NSPPolls: "Hypothèse Pécresse"), the candidates that set it apart from the
poll's other scenarios, and the full list of shares in each source.

In [5]:
desc = {2002: (crosscheck.describe_scenarios(fp), crosscheck.describe_scenarios(wp[wp["election"] == 2002])),
        2022: (crosscheck.describe_scenarios(nsp), crosscheck.describe_scenarios(wp[wp["election"] == 2022]))}
links_file = VALIDATION / "crosscheck_disagreements_links.csv"
doc_links = (worklists.read(links_file).set_index("poll_id")["document_to_use"].to_dict()
             if links_file.exists() else {})
polls_a = pd.concat([fp, nsp]).drop_duplicates("poll_id").set_index("poll_id")

def fmt(r):
    if not isinstance(r.candidate, str):
        return r.verdict
    va = r.share_a if r.status_a == "value" else r.status_a
    vb = r.share_b if r.status_b == "value" else r.status_b
    return f"{r.candidate}: {va} vs {vb}"

rows = []
for year, s in shares.items():
    da, db = desc[year]
    bad = s[s["verdict"].isin(TO_CHECK)]
    for (pa, pb, sa, sb), g in bad.groupby(["poll_id_a", "poll_id_b", "scenario_a", "scenario_b"], dropna=False):
        ref = da.loc[sa] if isinstance(sa, str) else db.loc[sb]
        rows.append(dict(
            election=year, check_id=f"{pa}|{sa if isinstance(sa, str) else sb}",
            poll_id=pa, poll_id_wikipedia=pb, institute=g["institute"].iloc[0],
            date_raw=polls_a.at[pa, "date_raw"],
            scenario=ref["scenario"], scenario_label=ref["scenario_label"],
            scenario_identified_by=ref["scenario_identified_by"],
            disagreements="; ".join(fmt(r) for r in g.itertuples()),
            shares_source_a=da.at[sa, "shares"] if isinstance(sa, str) else "",
            shares_wikipedia=db.at[sb, "shares"] if isinstance(sb, str) else "",
            document_to_use=doc_links.get(pa) or (polls_a.at[pa, "notice_url"] if isinstance(polls_a.at[pa, "notice_url"], str) else ""),
        ))
disagreements = pd.DataFrame(rows).sort_values(["election", "poll_id", "scenario"])
print(worklists.write(VALIDATION / "crosscheck_disagreements.csv", disagreements, key="check_id",
                      fill=["document_url", "correct_source", "notes"]))
disagreements.groupby("election").agg(scenarios=("check_id", "size"), polls=("poll_id", "nunique"))

written: crosscheck_disagreements.csv (79 rows to check)


,scenarios,polls
election,,
2002,19,19
2022,60,42


Polls found in only one source are listed in `data/interim/crosscheck_<year>_polls.csv`. In 2022 most
NSPPolls-only polls are rolling releases Wikipedia doesn't list:

In [6]:
m = matches[2022]
only_nsp = m[m["status"] == "only in a"]["poll_id_a"]
print(f"{len(only_nsp)} NSPPolls polls not on Wikipedia, of which rolling:",
      int(nsp.drop_duplicates("poll_id").set_index("poll_id").loc[only_nsp, "rolling"].eq(True).sum()))
print(f"{(m['status'] == 'only in b').sum()} Wikipedia polls not in NSPPolls")

104 NSPPolls polls not on Wikipedia, of which rolling: 90
16 Wikipedia polls not in NSPPolls


## 2 · Flagged overlaps (`overlap_client_unknown`)

In [7]:
resolved = pd.concat([crosscheck.resolve_flags(thinned, matches[y], wiki_side="b") for y in (2002, 2022)],
                     ignore_index=True)
resolved.to_csv(INTERIM / "flags_resolved.csv", index=False)
resolved.groupby(["election", "resolution"]).size().rename("polls").to_frame()

polls
election resolution                                                     
2002     independent poll (kept in the other source)                   6
2022     independent poll (kept in the other source)                  16
         rolling release: dropped by thinning in the other source      9

2017 has no second source, so its flagged polls (and any 2022 flag left without a match) go to
`validation/flagged_clients.csv`: **one row per poll**, including the polls they overlap. Fill in `client`
(who commissioned the poll, as named on the report's first page or in the Commission des sondages notice)
and `client_found_in` (the URL or document you read it in). Two overlapping polls are the same series
when they have the same pollster and the same client, so no separate yes/no is needed.

In [8]:
wp_polls = wp.drop_duplicates("poll_id").set_index("poll_id")
flag = thinned[thinned["overlap_client_unknown"]].set_index("poll_id")
unresolved = set(resolved.loc[resolved["resolution"].str.startswith("no match"), "poll_id"])
ids = set(flag.index[flag["election"] == 2017]) | unresolved
ids |= {flag.at[i, "overlaps_with"] for i in list(ids)}
# polls already resolved by hand stay on the list (their flag was lifted by notebook 02)
ids |= set(thinned.loc[thinned["client_from_validation"].astype(bool), "poll_id"])

def overlapping(pid):
    p = wp_polls.loc[pid]
    others = wp_polls[(wp_polls["election"] == p["election"]) & (wp_polls["institute"] == p["institute"])
                      & wp_polls.index.isin(ids) & (wp_polls.index != pid)]
    hit = others[(pd.to_datetime(others["fieldwork_start"]) <= pd.to_datetime(p["fieldwork_end"]))
                 & (pd.to_datetime(others["fieldwork_end"]) >= pd.to_datetime(p["fieldwork_start"]))]
    return " ; ".join(hit.index)

todo = wp_polls.loc[sorted(ids), ["election", "institute_raw", "date_raw", "fieldwork_start", "fieldwork_end",
                                  "sample_size", "upstream_source_url", "notice_url"]].reset_index()
todo.insert(4, "overlaps_with", todo["poll_id"].map(overlapping))
todo = todo.sort_values(["election", "institute_raw", "fieldwork_end"])
print(worklists.write(VALIDATION / "flagged_clients.csv", todo, fill=["client", "client_found_in", "notes"]))
todo[["election", "poll_id", "institute_raw", "date_raw", "overlaps_with"]]

kept existing flagged_clients.csv: 19/19 rows have answers


,election,poll_id,institute_raw,date_raw,overlaps_with
0,2017,2017_wp_harris-interactive_2017-04-05_1,Harris Interactive,3-5 avr.,2017_wp_harris-interactive_2017-04-06_1
1,2017,2017_wp_harris-interactive_2017-04-06_1,Harris Interactive,5-6 avr.,2017_wp_harris-interactive_2017-04-05_1
2,2017,2017_wp_ifop-fiducial_2017-02-17_1,Ifop-Fiducial,du 15 au 17,2017_wp_ifop-fiducial_2017-02-20_1
3,2017,2017_wp_ifop-fiducial_2017-02-20_1,Ifop-Fiducial,du 17 au 20,2017_wp_ifop-fiducial_2017-02-17_1 ; 2017_wp_ifop-fiduci...
4,2017,2017_wp_ifop-fiducial_2017-02-22_1,Ifop-Fiducial,du 20 au 22,2017_wp_ifop-fiducial_2017-02-20_1
5,2017,2017_wp_ifop-fiducial_2017-03-02_1,Ifop-Fiducial,28 fév. - 2 mars,2017_wp_ifop-fiducial_2017-03-04_1 ; 2017_wp_ifop-fiduci...
6,2017,2017_wp_ifop-fiducial_2017-03-04_1,Ifop-Fiducial,du 2 au 4,2017_wp_ifop-fiducial_2017-03-02_1 ; 2017_wp_ifop-fiduci...
7,2017,2017_wp_ifop-fiducial_2017-03-04_2,Ifop-Fiducial,du 1 au 4,2017_wp_ifop-fiducial_2017-03-02_1 ; 2017_wp_ifop-fiduci...
8,2017,2017_wp_ifop-fiducial_2017-03-07_1,Ifop-Fiducial,du 3 au 7,2017_wp_ifop-fiducial_2017-03-04_1 ; 2017_wp_ifop-fiduci...
9,2017,2017_wp_ifop-fiducial_2017-03-10_1,Ifop-Fiducial,du 7 au 10,2017_wp_ifop-fiducial_2017-03-07_1


## 3 · Random spot-check sample

For each election, 30 Wikipedia polls (from all polls, so the sample doesn't change when thinning decisions do), stratified by pollster (proportional, at least
one per pollster), drawn with the fixed seed. Check each against the original document: dates, sample
size, and every share (`shares_as_parsed` lists them by scenario). If no errors are found in 30 polls, the
error rate is below about 10 % at 95 % confidence (rule of three: 3/30).

In [9]:
cfg = yaml.safe_load((ROOT / "config" / "validation.yaml").read_text())
# All Wikipedia polls, not only those kept after thinning: the sample must not change when
# thinning decisions do, and transcription accuracy doesn't depend on thinning.
frame = wp.drop_duplicates("poll_id")
texts = worklists.shares_text(wp)

samples = []
for i, (year, g) in enumerate(frame.groupby("election")):
    s = sampling.draw(g, cfg["polls_per_election"], seed=cfg["seed"] + i)
    samples.append(s)
sample = pd.concat(samples)[["election", "poll_id", "institute", "institute_raw", "date_raw", "fieldwork_start",
                             "fieldwork_end", "reported_date", "sample_size", "upstream_source_url", "notice_url",
                             "table_label"]]
sample["shares_as_parsed"] = sample["poll_id"].map(texts)
print(worklists.write(VALIDATION / "spot_check_sample.csv", sample))
sample.groupby(["election", "institute"]).size().unstack(fill_value=0)

kept existing spot_check_sample.csv: 0/150 rows have answers


institute,Atlas Político,BVA,CSA,Cluster17,Dedicated Research,Elabe,Future Thinking - SSI,Harris Interactive,Ifop,Ipsos,Kantar Public,Kantar Sofres,LH2,Louis Harris,Odoxa,OpinionWay,Scan Research - Le Terrain,Sofres,TNS Sofres,YouGov
election,,,,,,,,,,,,,,,,,,,,
2002,0,3,7,0,0,0,0,0,5,6,0,0,0,2,0,0,0,7,0,0
2007,0,3,5,0,0,0,0,0,6,7,0,0,4,0,0,0,0,5,0,0
2012,0,3,3,0,0,0,0,3,8,5,0,0,2,0,0,4,0,2,0,0
2017,0,3,1,0,1,3,1,2,6,3,0,2,0,0,3,3,1,0,1,0
2022,1,2,0,2,0,3,0,4,5,3,1,0,0,0,2,6,0,0,0,1
